# Buscar Dados de Ordem de Serviços do MySQL

Este notebook conecta ao banco MySQL, executa um JOIN entre tabelas e exporta os dados para CSV.

## 1. Importar Bibliotecas

In [110]:
import pandas as pd
import mysql.connector
from mysql.connector import Error

## 2. Configurar Credenciais do Banco de Dados

In [111]:
# Configure suas credenciais aqui
config = {
    'host': 'localhost',
    'port': 3306,
    'user': 'root',  # Altere com seu usuário
    'password': '123456',  # Altere com sua senha
    'database': 'grotrack'  # Altere com o nome do seu banco
}

## 3. Conectar ao Banco de Dados

In [112]:
def conectar_mysql(config):
    """Conecta ao banco de dados MySQL"""
    try:
        conexao = mysql.connector.connect(**config)
        
        if conexao.is_connected():
            print("✓ Conexão estabelecida com sucesso!")
            return conexao
    
    except Error as erro:
        print(f"✗ Erro ao conectar ao MySQL: {erro}")
        return None

# Estabelecer conexão
conexao = conectar_mysql(config)

✓ Conexão estabelecida com sucesso!


## 4. Executar Query no Banco de Dados

In [113]:
def buscar_dados(conexao):
    """Executa a query e retorna os dados"""
    try:
        cursor = conexao.cursor(dictionary=True)
        
        query = """
        SELECT os.*, re.data_entrada_prevista, re.data_entrada_efetiva FROM ordem_de_servicos os 
        JOIN registro_entrada re
        ON re.fk_ordem_servico = os.id_ordem_servico
        WHERE os.status = 'FINALIZADO';
        """
        
        cursor.execute(query)
        dados = cursor.fetchall()
        cursor.close()
        
        query2 = """
        SELECT os.*, re.data_entrada_prevista, re.data_entrada_efetiva, s.* FROM ordem_de_servicos os 
        JOIN registro_entrada re
        ON re.fk_ordem_servico = os.id_ordem_servico
        JOIN itens_servicos s
        ON s.fk_ordem_servico = os.id_ordem_servico
        WHERE os.status = 'FINALIZADO';
        """

        dados2 = pd.read_sql(query2, conexao)

        print(f"✓ {len(dados)} registros recuperados")
        print(f"✓ {len(dados2)} registros recuperados")

        return dados, dados2
    
    except Error as erro:
        print(f"✗ Erro ao executar query: {erro}")
        return None

# Executar query
if conexao:
    dados = buscar_dados(conexao)[0]
    dados2 = buscar_dados(conexao)[1]

✓ 51 registros recuperados
✓ 90 registros recuperados
✓ 51 registros recuperados
✓ 90 registros recuperados


/tmp/ipykernel_16118/2105980990.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dados2 = pd.read_sql(query2, conexao)


## 5. Converter para DataFrame

In [114]:
# Converter para DataFrame
if dados:
    df = pd.DataFrame(dados)
    print(f"\n✓ DataFrame criado com {len(df)} linhas e {len(df.columns)} colunas")
    print(f"\nColunas: {list(df.columns)}")
    df2 = pd.DataFrame(dados2)
    print(f"\n✓ DataFrame 2 criado com {len(df2)} linhas e {len(df2.columns)} colunas")
    print(f"\nColunas: {list(df2.columns)}")
else:
    print("✗ Nenhum dado para converter")
    df = None
    df2 = None


✓ DataFrame criado com 51 linhas e 14 colunas

Colunas: ['id_ordem_servico', 'valor_total', 'valor_total_servicos', 'valor_total_produtos', 'data_saida_prevista', 'data_saida_efetiva', 'data_atualizacao', 'status', 'seguradora', 'nf_realizada', 'pagt_realizado', 'ativo', 'data_entrada_prevista', 'data_entrada_efetiva']

✓ DataFrame 2 criado com 90 linhas e 23 colunas

Colunas: ['id_ordem_servico', 'valor_total', 'valor_total_servicos', 'valor_total_produtos', 'data_saida_prevista', 'data_saida_efetiva', 'data_atualizacao', 'status', 'seguradora', 'nf_realizada', 'pagt_realizado', 'ativo', 'data_entrada_prevista', 'data_entrada_efetiva', 'id_registro_servico', 'fk_ordem_servico', 'preco_cobrado', 'parte_veiculo', 'lado_veiculo', 'tipo_servico', 'cor', 'especificacao_servico', 'tipo_pintura']


## 6. Visualizar Dados

In [115]:
# Visualizar primeiras linhas
if df is not None and df2 is not None:
    display(df.head())
    display(df2.head())

,id_ordem_servico,valor_total,valor_total_servicos,valor_total_produtos,data_saida_prevista,data_saida_efetiva,data_atualizacao,status,seguradora,nf_realizada,pagt_realizado,ativo,data_entrada_prevista,data_entrada_efetiva
0,6,2079.60,1400.00,679.60,2026-03-20,2026-03-21,2026-05-18,FINALIZADO,1,1,1,1,2026-03-15,2026-03-15
1,7,635.00,470.00,165.00,2026-03-22,2026-03-22,2026-05-18,FINALIZADO,0,0,1,1,2026-03-16,2026-03-16
2,14,522.00,450.00,72.00,2026-04-05,2026-04-06,2026-05-18,FINALIZADO,1,1,1,1,2026-04-03,2026-04-04
3,15,414.00,390.00,24.00,2026-04-04,2026-04-05,2026-05-18,FINALIZADO,0,0,1,1,2026-04-02,2026-04-03
4,16,839.70,720.00,119.70,2026-04-03,2026-04-04,2026-05-18,FINALIZADO,1,0,0,1,2026-04-01,2026-04-02


,id_ordem_servico,valor_total,valor_total_servicos,valor_total_produtos,data_saida_prevista,data_saida_efetiva,data_atualizacao,status,seguradora,nf_realizada,...,data_entrada_efetiva,id_registro_servico,fk_ordem_servico,preco_cobrado,parte_veiculo,lado_veiculo,tipo_servico,cor,especificacao_servico,tipo_pintura
0,6,2079.6,1400.0,679.6,2026-03-20,2026-03-21,2026-05-18,FINALIZADO,1,1,...,2026-03-15,93,6,840.0,CURVAO,DIANTEIRO_ESQUERDO,PINTURA,Azul,Pintura parcial na coluna frontal esquerda,PARCIAL
1,6,2079.6,1400.0,679.6,2026-03-20,2026-03-21,2026-05-18,FINALIZADO,1,1,...,2026-03-15,94,6,560.0,PAINEL,COMPLETO,MECANICA,NaN,Revisão e ajuste de painel elétrico interno,NAO_APLICAVEL
2,7,635.0,470.0,165.0,2026-03-22,2026-03-22,2026-05-18,FINALIZADO,0,0,...,2026-03-16,95,7,470.0,GRADE,DIANTEIRO,FUNILARIA,Cinza,Troca e ajuste de grade dianteira,NAO_APLICAVEL
3,14,522.0,450.0,72.0,2026-04-05,2026-04-06,2026-05-18,FINALIZADO,1,1,...,2026-04-04,106,14,450.0,PAINEL,COMPLETO,MECANICA,NaN,Ajuste de painel eletrico,NAO_APLICAVEL
4,15,414.0,390.0,24.0,2026-04-04,2026-04-05,2026-05-18,FINALIZADO,0,0,...,2026-04-03,107,15,390.0,GRADE,DIANTEIRO,FUNILARIA,Preto,Troca e fixacao de grade,NAO_APLICAVEL


## 7. Informações do DataFrame

In [116]:
# Informações sobre o DataFrame
if df is not None and df2 is not None:
    print(df.info())
    print("\nEstatísticas:")
    display(df.describe())
    print(df2.info())
    print("\nEstatísticas 2:")
    display(df2.describe())

<class 'pandas.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_ordem_servico       51 non-null     int64 
 1   valor_total            51 non-null     object
 2   valor_total_servicos   51 non-null     object
 3   valor_total_produtos   51 non-null     object
 4   data_saida_prevista    51 non-null     object
 5   data_saida_efetiva     51 non-null     object
 6   data_atualizacao       51 non-null     object
 7   status                 51 non-null     str   
 8   seguradora             51 non-null     int64 
 9   nf_realizada           51 non-null     int64 
 10  pagt_realizado         51 non-null     int64 
 11  ativo                  51 non-null     int64 
 12  data_entrada_prevista  51 non-null     object
 13  data_entrada_efetiva   51 non-null     object
dtypes: int64(5), object(8), str(1)
memory usage: 5.7+ KB
None

Estatísticas:


,id_ordem_servico,seguradora,nf_realizada,pagt_realizado,ativo
count,51.000000,51.000000,51.000000,51.000000,51.000000
mean,58.274510,0.588235,0.725490,0.901961,0.980392
std,26.659391,0.497050,0.450708,0.300327,0.140028
min,6.000000,0.000000,0.000000,0.000000,0.000000
25%,39.000000,0.000000,0.000000,1.000000,1.000000
50%,61.000000,1.000000,1.000000,1.000000,1.000000
75%,81.000000,1.000000,1.000000,1.000000,1.000000
max,97.000000,1.000000,1.000000,1.000000,1.000000


<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id_ordem_servico       90 non-null     int64  
 1   valor_total            90 non-null     float64
 2   valor_total_servicos   90 non-null     float64
 3   valor_total_produtos   90 non-null     float64
 4   data_saida_prevista    90 non-null     object 
 5   data_saida_efetiva     90 non-null     object 
 6   data_atualizacao       90 non-null     object 
 7   status                 90 non-null     str    
 8   seguradora             90 non-null     int64  
 9   nf_realizada           90 non-null     int64  
 10  pagt_realizado         90 non-null     int64  
 11  ativo                  90 non-null     int64  
 12  data_entrada_prevista  90 non-null     object 
 13  data_entrada_efetiva   90 non-null     object 
 14  id_registro_servico    90 non-null     int64  
 15  fk_ordem_servico   

,id_ordem_servico,valor_total,valor_total_servicos,valor_total_produtos,seguradora,nf_realizada,pagt_realizado,ativo,id_registro_servico,fk_ordem_servico,preco_cobrado
count,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000
mean,62.100000,1365.973333,1324.888889,41.084444,0.588889,0.777778,0.900000,0.988889,166.488889,62.100000,700.555556
std,24.936797,395.921466,384.506818,122.560889,0.494792,0.418069,0.301681,0.105409,32.677603,24.936797,183.978266
min,6.000000,414.000000,390.000000,0.000000,0.000000,0.000000,0.000000,0.000000,93.000000,6.000000,280.000000
25%,45.750000,1097.500000,1032.500000,0.000000,0.000000,1.000000,1.000000,1.000000,143.500000,45.750000,552.500000
50%,63.000000,1395.000000,1385.000000,0.000000,1.000000,1.000000,1.000000,1.000000,169.500000,63.000000,710.000000
75%,83.000000,1622.500000,1597.500000,0.000000,1.000000,1.000000,1.000000,1.000000,193.750000,83.000000,865.000000
max,97.000000,2290.000000,1930.000000,679.600000,1.000000,1.000000,1.000000,1.000000,216.000000,97.000000,990.000000


## 8. Exportar para CSV

In [117]:
def gerar_csv(df, nome_arquivo):
    """Exporta DataFrame para arquivo CSV"""
    try:
        df.to_csv(nome_arquivo, index=False, encoding='utf-8')
        print(f"✓ Arquivo '{nome_arquivo}' gerado com sucesso!")
        return True
    except Exception as erro:
        print(f"✗ Erro ao gerar CSV: {erro}")
        return False

# Gerar CSV
if df is not None and df2 is not None:
    gerar_csv(df, '../refined/grafana/os_data.csv')
    gerar_csv(df2, '../refined/grafana/os_data_com_servicos.csv')

✓ Arquivo '../refined/grafana/os_data.csv' gerado com sucesso!
✓ Arquivo '../refined/grafana/os_data_com_servicos.csv' gerado com sucesso!


## 9. Fechar Conexão

In [118]:
# Fechar conexão
if conexao and conexao.is_connected():
    conexao.close()
    print("✓ Conexão fechada.")

✓ Conexão fechada.
